# Hamiltonian Simulation

## **Quantum simulation** — building a quantum system that mimics another quantum system — was Feynman's original 1982 motivation for quantum computers. The fundamental task is: given a Hamiltonian $H$ and a time $t$, implement (approximately) the unitary $e^{-iHt}$.

# 1. Why this is the killer app

## Classical simulation of an $n$-body quantum system requires storing a state vector of dimension $2^n$. For $n = 60$ this is already beyond classical reach. Quantum simulation, by contrast, uses $n$ qubits to simulate $n$ particles — *linear* in the size of the system. This is widely regarded as the most likely first source of quantum advantage in chemistry and materials science.

# 2. Trotter-Suzuki formulas

## If $H = \sum_k H_k$ where each $e^{-i H_k t}$ is easy to implement,

$$ \Large  e^{-iHt} \approx \Big(\prod_k e^{-iH_k t/r}\Big)^r $$

## for large $r$. This is the **first-order Trotter formula**, with error $O(t^2 / r)$ per step. Higher-order Trotter-Suzuki formulas give better error scaling at the cost of more sub-steps per Trotter step.

## Trotterisation is what most demonstrations on real hardware actually use.

In [1]:
# Compare exact e^{-iHt} with a first-order Trotter approximation
import numpy as np
from scipy.linalg import expm

X = np.array([[0,1],[1,0]], dtype=complex)
Z = np.array([[1,0],[0,-1]], dtype=complex)
I = np.eye(2, dtype=complex)

# H = X o I + I o Z + 0.5 * Z o Z
H = np.kron(X, I) + np.kron(I, Z) + 0.5*np.kron(Z, Z)
terms = [np.kron(X, I), np.kron(I, Z), 0.5*np.kron(Z, Z)]
t = 1.0

exact = expm(-1j*H*t)
for r in [1, 10, 100, 1000]:
    step = np.eye(4, dtype=complex)
    for term in terms:
        step = step @ expm(-1j*term*t/r)
    trotter = np.linalg.matrix_power(step, r)
    err = np.linalg.norm(trotter - exact)
    print(f'r = {r:5d}: Trotter error = {err:.6f}')

r =     1: Trotter error = 0.869290
r =    10: Trotter error = 0.080494
r =   100: Trotter error = 0.008043
r =  1000: Trotter error = 0.000804


# 3. Modern simulation algorithms

## - **Linear combination of unitaries (LCU)**: implement $e^{-iHt}$ via a truncated Taylor expansion of unitary primitives.
## - **Qubitisation / quantum signal processing**: optimal-complexity Hamiltonian simulation, $O(t \log(1/\varepsilon))$   queries to a block encoding of $H$.
## - **Random Trotter** (Childs–Wiebe): randomise the order of $H_k$ for second-order accuracy with first-order circuits.

## Each pushes the constant prefactors closer to the lower bounds and is highly relevant for resource estimation in chemistry.

# 4. Applications

## - **Quantum chemistry**: molecular ground states and dynamics.
## - **Condensed-matter physics**: Hubbard model, lattice gauge theories.
## - **Nuclear physics**: lattice QCD simulations.
## - **High-energy physics**: scattering amplitudes via simulation.

# Recap

## - Goal: implement $e^{-iHt}$ for a given Hamiltonian $H$.
## - **Trotter-Suzuki** is the workhorse; modern algorithms (qubitisation, QSP) push complexity to optimum.
## - Likely the first place quantum computers will outperform classical, especially in chemistry/materials.

## **End of chapter 7.** Next: error correction — the other prerequisite for fault-tolerant quantum computing.